# Trabajo Final — Pipeline de pedidos de TiendaBSG
**Tópicos Avanzados para Data Science en Python**

**Alumno(a):** Yuri Jean Pool Peña Bilbao
**Repositorio público:** **https://github.com/JEANPOOLPB/trabajo-final-spark**

---

Eres el *data engineer* de **TiendaBSG**, una tienda online con despachos en 6 ciudades del Perú. Cada día el sistema de ventas deja un archivo CSV con los pedidos del día en la carpeta `datos/raw/`. Tu trabajo es procesar los pedidos de **octubre 2026** de principio a fin:

| Parte | Qué harás | Capítulo | Puntos |
|---|---|---|---|
| 1. Revisar los archivos | Explorar `raw/`, separar archivos vacíos y contar registros | 1. Línea de comandos | 3.5 |
| 2. Analizar con Spark | Limpiar, filtrar, agrupar y contar con DataFrames o Spark SQL | 3. Spark | 12 |
| 3. Cerrar el proceso | Mover los CSV procesados y dejar el resultado final listo | 1. Línea de comandos | 2 |
| 4. Entregar | Subir este notebook y el resultado a tu repositorio público | 2. Git | 2.5 |
| | | **Total** | **20** |

**Reglas**
- Ejecuta las celdas **en orden**. Si algo se rompe, vuelve a ejecutar la **Preparación** (reinicia los datos) y continúa.
- En la Parte 2 usa solo **DataFrames** (API de PySpark o Spark SQL). **No uses RDDs.**
- Responde las preguntas como **comentarios** (`# ...`) en tu propia celda de código.
- Antes de descargar el notebook: **Entorno de ejecución → Reiniciar y ejecutar todo**, para que se vean todas las salidas en GitHub.

**Cómo usar esta versión:** lee los comentarios y ejecuta cada celda en orden. Las respuestas numéricas corresponden al generador original con semilla 2026. Completa el enlace de GitHub antes de entregar.

**Fuente:** [plantilla original del docente](https://github.com/bvivanco/topicos_avanzados_ds/blob/main/Trabajo_Final.ipynb).

## Preparación (no se califica)
Ejecuta estas tres celdas. Instalan PySpark, crean la sesión de Spark y generan los archivos del mes:

```
/content/trabajo_final/
├── datos/
│   ├── raw/          ← 31 archivos pedidos_2026-10-DD.csv (uno por día)
│   ├── procesados/   ← vacía: aquí moverás los CSV ya procesados (Parte 3)
│   └── cuarentena/   ← vacía: aquí moverás los archivos con problemas (Parte 1)
└── resultados/       ← vacía: aquí guardarás el resumen final (Partes 2 y 3)
```

In [1]:
# Java + PySpark (tarda ~1 minuto la primera vez)
import shutil
if shutil.which("java") is None:
    !apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q uninstall -y dataproc-spark-connect  # paquete de Colab que pide PySpark 4; no lo usamos
!pip -q install "pyspark==3.5.*"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
         .master("local[*]")
         .appName("BSG-TrabajoFinal")
         .config("spark.sql.shuffle.partitions", "8")          # suficiente para este conjunto pequeño
         .config("spark.ui.showConsoleProgress", "false")   # sin barras de progreso
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                     # oculta los logs de la JVM
print("Versión de Spark:", spark.version)

Versión de Spark: 3.5.9


In [3]:
# Genera los pedidos de octubre 2026 (volver a ejecutar = reiniciar los datos)
import csv, os, random
from datetime import date

TF = "/content/trabajo_final"
RAW = f"{TF}/datos/raw"
PROCESADOS = f"{TF}/datos/procesados"
CUARENTENA = f"{TF}/datos/cuarentena"
RESULTADOS = f"{TF}/resultados"

shutil.rmtree(TF, ignore_errors=True)
for carpeta in (RAW, PROCESADOS, CUARENTENA, RESULTADOS):
    os.makedirs(carpeta)

rng = random.Random(2026)
CIUDADES = {"Lima": 40, "Arequipa": 15, "Trujillo": 12, "Chiclayo": 10, "Cusco": 13, "Piura": 10}
CATALOGO = {  # producto: (categoria, precio de lista, peso en las ventas)
    "Laptop": ("Tecnologia", 3200.0, 8), "Celular": ("Tecnologia", 1500.0, 15),
    "Audifonos": ("Tecnologia", 180.0, 20), "Licuadora": ("Hogar", 250.0, 12),
    "Cafetera": ("Hogar", 320.0, 8), "Zapatillas": ("Moda", 280.0, 22),
    "Casaca": ("Moda", 220.0, 15)}
DEVOLUCION = {"Tecnologia": 4, "Hogar": 3, "Moda": 12}  # % de pedidos devueltos
PAGOS = {"Tarjeta": 40, "Yape": 30, "Efectivo": 15, "Transferencia": 15}
DIAS_VACIOS = {12, 25}  # el sistema falló y dejó estos archivos vacíos

id_pedido, dia_anterior = 0, []
for dia in range(1, 32):
    fecha = date(2026, 10, dia).isoformat()
    ruta = f"{RAW}/pedidos_{fecha}.csv"
    if dia in DIAS_VACIOS:
        open(ruta, "w").close()
        continue
    filas = []
    for _ in range(rng.randint(1500, 2500)):
        id_pedido += 1
        ciudad = rng.choices(list(CIUDADES), weights=CIUDADES.values())[0]
        producto = rng.choices(list(CATALOGO), weights=[v[2] for v in CATALOGO.values()])[0]
        categoria, precio_lista, _ = CATALOGO[producto]
        precio = round(precio_lista * rng.uniform(0.9, 1.1), 2)
        cantidad = rng.randint(1, 4)
        devolucion = DEVOLUCION[categoria]
        estado = rng.choices(["ENTREGADO", "CANCELADO", "DEVUELTO"],
                             weights=[90 - devolucion, 10, devolucion])[0]
        pago = rng.choices(list(PAGOS), weights=PAGOS.values())[0]
        # Errores típicos del sistema de origen
        r = rng.random()
        if r < 0.10:
            ciudad = ciudad.upper()
        elif r < 0.20:
            ciudad = f" {ciudad.lower()}  "
        if rng.random() < 0.05:
            estado = estado.lower()
        if rng.random() < 0.02:
            cantidad = ""
        if rng.random() < 0.01:
            precio = -precio
        filas.append([id_pedido, fecha, ciudad, categoria, producto, cantidad, precio, pago, estado])
    duplicados = rng.sample(dia_anterior, len(dia_anterior) // 50)  # pedidos reenviados
    dia_anterior = filas
    with open(ruta, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f, lineterminator="\n")
        w.writerow(["id_pedido", "fecha", "ciudad", "categoria", "producto",
                    "cantidad", "precio_unitario", "metodo_pago", "estado"])
        w.writerows(filas + duplicados)

%cd {TF}
!ls datos/raw | head -3
print("...")

/content/trabajo_final
pedidos_2026-10-01.csv
pedidos_2026-10-02.csv
pedidos_2026-10-03.csv
...


## Parte 1 — Revisar los archivos (Línea de comandos · 3.5 pts)
Antes de procesar nada, un *data engineer* revisa qué llegó. Usa **solo comandos de terminal** (`!comando` para una línea, o `%%bash` al inicio de la celda para varias). Ya estás ubicado en `/content/trabajo_final`.

| Comando | ¿Qué hace? |
|---|---|
| `ls -lh carpeta` | Lista archivos con su tamaño |
| `ls carpeta \| wc -l` | Cuenta cuántos archivos hay |
| `du -sh carpeta` | Tamaño total de una carpeta |
| `head -n 5 archivo` | Muestra las primeras líneas |
| `find carpeta -type f -empty` | Busca archivos vacíos |
| `mv origen destino` | Mueve (o renombra) archivos |
| `wc -l archivos` | Cuenta líneas |
| `grep -ci texto` | Cuenta líneas que contienen un texto (sin importar mayúsculas) |

### ✏️ Ejercicio 1.1 — Explorar la carpeta raw (1 pts)
1. Lista los archivos de `datos/raw` con su tamaño.
2. Cuenta cuántos archivos hay y el tamaño total de la carpeta.
3. Muestra las 5 primeras líneas del archivo del 1 de octubre.
4. Responde: ¿qué columnas tiene cada archivo? ¿Ves algún archivo raro en el listado?

In [4]:
%%bash
# ls lista archivos; -lh añade detalles y tamaños legibles.
ls -lh datos/raw

# wc -l cuenta las líneas del listado: una por archivo.
ls datos/raw | wc -l
du -sh datos/raw
head -n 5 datos/raw/pedidos_2026-10-01.csv

# Respuesta: hay 9 columnas: id_pedido, fecha, ciudad, categoria,
# producto, cantidad, precio_unitario, metodo_pago y estado.
# Hay 31 archivos. Los del 12 y 25 de octubre tienen tamaño 0 bytes.

total 3.8M
-rw-r--r-- 1 root root 106K Nov 13 00:22 pedidos_2026-10-01.csv
-rw-r--r-- 1 root root 138K Nov 13 00:22 pedidos_2026-10-02.csv
-rw-r--r-- 1 root root 103K Nov 13 00:22 pedidos_2026-10-03.csv
-rw-r--r-- 1 root root 127K Nov 13 00:22 pedidos_2026-10-04.csv
-rw-r--r-- 1 root root 165K Nov 13 00:22 pedidos_2026-10-05.csv
-rw-r--r-- 1 root root 113K Nov 13 00:22 pedidos_2026-10-06.csv
-rw-r--r-- 1 root root 140K Nov 13 00:22 pedidos_2026-10-07.csv
-rw-r--r-- 1 root root 166K Nov 13 00:22 pedidos_2026-10-08.csv
-rw-r--r-- 1 root root 112K Nov 13 00:22 pedidos_2026-10-09.csv
-rw-r--r-- 1 root root 115K Nov 13 00:22 pedidos_2026-10-10.csv
-rw-r--r-- 1 root root 135K Nov 13 00:22 pedidos_2026-10-11.csv
-rw-r--r-- 1 root root    0 Nov 13 00:22 pedidos_2026-10-12.csv
-rw-r--r-- 1 root root 109K Nov 13 00:22 pedidos_2026-10-13.csv
-rw-r--r-- 1 root root 135K Nov 13 00:22 pedidos_2026-10-14.csv
-rw-r--r-- 1 root root 117K Nov 13 00:22 pedidos_2026-10-15.csv
-rw-r--r-- 1 root root 137K N

### ✏️ Ejercicio 1.2 — Separar los archivos vacíos (1.5 pts)
1. Encuentra los archivos vacíos de `datos/raw` con `find`.
2. Muévelos a `datos/cuarentena` (con `mv` o con `find ... -exec mv {} destino \;`).
3. Comprueba: lista `datos/cuarentena` y cuenta cuántos archivos quedan en `datos/raw`.
4. Responde: ¿por qué conviene apartarlos en lugar de borrarlos?

In [5]:
%%bash
# Primero identificamos los archivos vacíos; luego los movemos.
find datos/raw -type f -name '*.csv' -empty
find datos/raw -type f -name '*.csv' -empty -exec mv {} datos/cuarentena/ \;

ls -lh datos/cuarentena
find datos/cuarentena -type f -name '*.csv' | wc -l
find datos/raw -type f -name '*.csv' | wc -l

# Respuesta: quedan 2 archivos en cuarentena y 29 en raw.
# Se apartan para conservar evidencia del fallo y poder revisarlos
# o solicitar su reenvío. Borrarlos haría perder esa trazabilidad.

datos/raw/pedidos_2026-10-12.csv
datos/raw/pedidos_2026-10-25.csv
total 0
-rw-r--r-- 1 root root 0 Nov 13 00:22 pedidos_2026-10-12.csv
-rw-r--r-- 1 root root 0 Nov 13 00:22 pedidos_2026-10-25.csv
2
29


### ✏️ Ejercicio 1.3 — Contar registros antes de Spark (1 pts)
1. Cuenta las líneas de todos los CSV de `datos/raw` juntos con `wc -l` (la última línea muestra el total).
2. Calcula cuántas **filas de datos** hay: total de líneas menos una cabecera por archivo. Anota el número: lo compararás en el Ejercicio 2.1.
3. Con `grep`, cuenta cuántas líneas contienen la palabra `cancelado` sin importar mayúsculas.

In [6]:
%%bash
# Este listado incluye una cabecera por cada CSV.
wc -l datos/raw/*.csv

# cat reúne el contenido; wc obtiene el total de líneas.
lineas_totales=$(cat datos/raw/*.csv | wc -l)
numero_archivos=$(find datos/raw -type f -name '*.csv' | wc -l)
filas_datos=$((lineas_totales - numero_archivos))
printf 'Líneas totales: %s\nCabeceras: %s\nFilas de datos: %s\n' \
  "$lineas_totales" "$numero_archivos" "$filas_datos"

# -i ignora mayúsculas; -c cuenta las líneas coincidentes.
cat datos/raw/*.csv | grep -ci 'cancelado'

# Respuesta: 57 332 líneas - 29 cabeceras = 57 303 filas de datos.
# Hay 5 616 líneas con cancelado, antes de limpiar y quitar duplicados.
# Aquí cada registro ocupa una línea: son los CSV del generador del curso.

   1622 datos/raw/pedidos_2026-10-01.csv
   2096 datos/raw/pedidos_2026-10-02.csv
   1557 datos/raw/pedidos_2026-10-03.csv
   1936 datos/raw/pedidos_2026-10-04.csv
   2512 datos/raw/pedidos_2026-10-05.csv
   1698 datos/raw/pedidos_2026-10-06.csv
   2090 datos/raw/pedidos_2026-10-07.csv
   2492 datos/raw/pedidos_2026-10-08.csv
   1681 datos/raw/pedidos_2026-10-09.csv
   1714 datos/raw/pedidos_2026-10-10.csv
   2016 datos/raw/pedidos_2026-10-11.csv
   1640 datos/raw/pedidos_2026-10-13.csv
   2012 datos/raw/pedidos_2026-10-14.csv
   1758 datos/raw/pedidos_2026-10-15.csv
   2058 datos/raw/pedidos_2026-10-16.csv
   1970 datos/raw/pedidos_2026-10-17.csv
   2048 datos/raw/pedidos_2026-10-18.csv
   1832 datos/raw/pedidos_2026-10-19.csv
   2264 datos/raw/pedidos_2026-10-20.csv
   2144 datos/raw/pedidos_2026-10-21.csv
   1824 datos/raw/pedidos_2026-10-22.csv
   2312 datos/raw/pedidos_2026-10-23.csv
   2171 datos/raw/pedidos_2026-10-24.csv
   1716 datos/raw/pedidos_2026-10-26.csv
   2520 datos/ra

## Parte 2 — Analizar con Spark (DataFrames · 12 pts)
Cada ejercicio puedes resolverlo con la **API de DataFrames** o con **Spark SQL**; el Ejercicio 2.7 es obligatoriamente con SQL. **No uses RDDs.**

| API de DataFrames | Spark SQL | ¿Qué hace? |
|---|---|---|
| `df.select("a", "b")` | `SELECT a, b` | Elegir columnas |
| `df.filter(F.col("a") > 0)` | `WHERE a > 0` | Filtrar filas |
| `df.dropDuplicates(["id"])` | `SELECT DISTINCT ...` | Quitar duplicados |
| `df.dropna(subset=["a"])` | `WHERE a IS NOT NULL` | Quitar nulos |
| `df.withColumn("t", expr)` | `SELECT ..., expr AS t` | Crear o reemplazar una columna |
| `df.groupBy("a").agg(F.sum("b"))` | `GROUP BY a` | Agrupar y agregar |
| `df.groupBy("a").count()` | `COUNT(*)` | Contar por grupo |
| `df.orderBy(F.desc("b"))` | `ORDER BY b DESC` | Ordenar |
| `df.createOrReplaceTempView("v")` | `spark.sql("SELECT ... FROM v")` | Usar SQL sobre un DataFrame |

> 💡 Para que las sumas grandes no salgan en notación científica: `.cast("decimal(18,2)")` o `CAST(SUM(x) AS DECIMAL(18,2))`.

### ✏️ Ejercicio 2.1 — Leer los pedidos (1 pts)
1. Lee **todos** los CSV de `datos/raw` en un solo DataFrame llamado `pedidos` (pista: `spark.read.csv(f"{RAW}/*.csv", header=True, inferSchema=True)`).
2. Muestra el esquema con `printSchema()` y cuenta las filas.
3. Responde: ¿coincide con el número de filas de datos que calculaste en el Ejercicio 1.3?

In [7]:
# header=True reconoce las cabeceras; inferSchema=True infiere los tipos.
pedidos = spark.read.csv(f"{RAW}/*.csv", header=True, inferSchema=True)
pedidos.printSchema()
n_antes = pedidos.count()
print("Filas leídas por Spark:", n_antes)

# Respuesta: sí, Spark lee 57 303 filas, igual que el ejercicio 1.3.

root
 |-- id_pedido: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- producto: string (nullable = true)
 |-- cantidad: integer (nullable = true)
 |-- precio_unitario: double (nullable = true)
 |-- metodo_pago: string (nullable = true)
 |-- estado: string (nullable = true)

Filas leídas por Spark: 57303


### ✏️ Ejercicio 2.2 — Explorar la calidad de los datos (1.5 pts)
1. Muestra 5 filas solo de las columnas `fecha`, `ciudad`, `producto`, `cantidad` y `estado`.
2. Muestra los valores distintos de `ciudad` y cuéntalos. ¿Por qué hay más de 6?
3. Cuenta cuántas filas tienen `cantidad` nula y cuántas tienen `precio_unitario` menor o igual a 0.
4. Cuenta cuántos `id_pedido` están repetidos (filas totales menos `id_pedido` distintos).

In [8]:
pedidos.select("fecha", "ciudad", "producto", "cantidad", "estado").show(5, truncate=False)

ciudades_originales = pedidos.select("ciudad").distinct()
ciudades_originales.orderBy("ciudad").show(30, truncate=False)
print("Valores distintos de ciudad:", ciudades_originales.count())

cantidad_nula = pedidos.filter(F.col("cantidad").isNull()).count()
precio_invalido = pedidos.filter(F.col("precio_unitario") <= 0).count()
ids_distintos = pedidos.select("id_pedido").distinct().count()
filas_repetidas = n_antes - ids_distintos
print("Cantidad nula:", cantidad_nula)
print("Precio <= 0:", precio_invalido)
print("IDs distintos:", ids_distintos)
print("Filas repetidas adicionales:", filas_repetidas)

# Respuesta: aparecen 18 formas de escribir las mismas 6 ciudades,
# por diferencias de mayúsculas/minúsculas y espacios alrededor.
# Hay 1 127 cantidades nulas, 582 precios inválidos y 1 068 filas
# repetidas adicionales. Estos problemas se miden sobre los datos originales
# y pueden coincidir en una fila; no deben sumarse como pérdidas separadas.

+----------+--------+----------+--------+---------+
|fecha     |ciudad  |producto  |cantidad|estado   |
+----------+--------+----------+--------+---------+
|2026-10-27|Lima    |Casaca    |2       |ENTREGADO|
|2026-10-27|Lima    |Zapatillas|1       |CANCELADO|
|2026-10-27|Trujillo|Cafetera  |4       |ENTREGADO|
|2026-10-27|CUSCO   |Audifonos |4       |ENTREGADO|
|2026-10-27|Arequipa|Audifonos |4       |ENTREGADO|
+----------+--------+----------+--------+---------+
only showing top 5 rows

+-----------+
|ciudad     |
+-----------+
| arequipa  |
| chiclayo  |
| cusco     |
| lima      |
| piura     |
| trujillo  |
|AREQUIPA   |
|Arequipa   |
|CHICLAYO   |
|CUSCO      |
|Chiclayo   |
|Cusco      |
|LIMA       |
|Lima       |
|PIURA      |
|Piura      |
|TRUJILLO   |
|Trujillo   |
+-----------+

Valores distintos de ciudad: 18
Cantidad nula: 1127
Precio <= 0: 582
IDs distintos: 56235
Filas repetidas adicionales: 1068


### ✏️ Ejercicio 2.3 — Limpiar los pedidos (2 pts)
Crea el DataFrame `df_limpio` a partir de `pedidos`:
1. Elimina los duplicados por `id_pedido`.
2. Elimina las filas con `cantidad` nula y con `precio_unitario <= 0`.
3. Normaliza `ciudad` (quita espacios y deja la primera letra en mayúscula: `F.initcap(F.trim(...))`).
4. Normaliza `estado` en mayúsculas y sin espacios (`F.upper(F.trim(...))`).
5. Agrega la columna `total = cantidad * precio_unitario` redondeada a 2 decimales.

Muestra cuántas filas había antes y después, y los valores distintos de `ciudad` (deben ser 6) y de `estado` (deben ser 3).

In [9]:
# Aplicamos las reglas en el orden pedido para explicar cada pérdida de filas.
sin_duplicados = pedidos.dropDuplicates(["id_pedido"])
sin_nulos = sin_duplicados.dropna(subset=["cantidad"])
validos = sin_nulos.filter(F.col("precio_unitario") > 0)

df_limpio = (validos
    .withColumn("ciudad", F.initcap(F.trim(F.col("ciudad"))))
    .withColumn("estado", F.upper(F.trim(F.col("estado"))))
    .withColumn("total", F.round(
        F.col("cantidad") * F.col("precio_unitario").cast("decimal(12,2)"), 2
    ).cast("decimal(18,2)")))
# Decimal representa importes con dos decimales; total es el importe del pedido.

n_limpios = df_limpio.count()
print("Antes:", n_antes)
print("Después de quitar duplicados:", sin_duplicados.count())
print("Después de quitar cantidades nulas:", sin_nulos.count())
print("Después de quitar precios inválidos:", n_limpios)
print("Filas retiradas en total:", n_antes - n_limpios)
df_limpio.select("ciudad").distinct().orderBy("ciudad").show()
df_limpio.select("estado").distinct().orderBy("estado").show()

# Respuesta: 57 303 -> 56 235 -> 55 126 -> 54 566 filas.
# Se retiran, en ese orden, 1 068 duplicados, 1 109 nulos y 560 precios
# inválidos. Quedan 6 ciudades y 3 estados: CANCELADO, DEVUELTO y ENTREGADO.
# No se inventan cantidades ni se convierten precios negativos en positivos.

Antes: 57303
Después de quitar duplicados: 56235
Después de quitar cantidades nulas: 55126
Después de quitar precios inválidos: 54566
Filas retiradas en total: 2737
+--------+
|  ciudad|
+--------+
|Arequipa|
|Chiclayo|
|   Cusco|
|    Lima|
|   Piura|
|Trujillo|
+--------+

+---------+
|   estado|
+---------+
|CANCELADO|
| DEVUELTO|
|ENTREGADO|
+---------+



### ✏️ Ejercicio 2.4 — Filtrar pedidos (1.5 pts)
1. Crea el DataFrame `entregados` solo con los pedidos en estado `ENTREGADO` y cuéntalos.
2. ¿Cuántos pedidos entregados en **Lima**, pagados con **Yape**, tienen un `total` mayor a **1000**?
3. Muestra los 5 pedidos entregados de mayor `total` (columnas `id_pedido`, `ciudad`, `producto`, `cantidad`, `total`).

In [10]:
entregados = df_limpio.filter(F.col("estado") == "ENTREGADO")
print("Pedidos entregados:", entregados.count())
3
# & exige que se cumplan las tres condiciones a la vez.
lima_yape = entregados.filter(
    (F.col("ciudad") == "Lima") &
    (F.col("metodo_pago") == "Yape") &
    (F.col("total") > 1000)
)
print("Entregados en Lima con Yape y total > 1000:", lima_yape.count())

(entregados.select("id_pedido", "ciudad", "producto", "cantidad", "total")
    .orderBy(F.desc("total"), "id_pedido")
    .show(5, truncate=False))

# Respuesta: hay 45 665 entregados; 1 812 cumplen el filtro Lima/Yape/>1000.
# El mayor pedido entregado es el 52149, por S/ 14 076.40.

Pedidos entregados: 45665
Entregados en Lima con Yape y total > 1000: 1812
+---------+--------+--------+--------+--------+
|id_pedido|ciudad  |producto|cantidad|total   |
+---------+--------+--------+--------+--------+
|52149    |Chiclayo|Laptop  |4       |14076.40|
|47253    |Chiclayo|Laptop  |4       |14071.40|
|12957    |Lima    |Laptop  |4       |14070.04|
|45636    |Arequipa|Laptop  |4       |14065.44|
|55798    |Piura   |Laptop  |4       |14064.44|
+---------+--------+--------+--------+--------+
only showing top 5 rows



### ✏️ Ejercicio 2.5 — Conteos (1.5 pts)
1. Cuenta los pedidos por `estado`, de mayor a menor.
2. Cuenta los pedidos por `metodo_pago`, de mayor a menor.
3. Responde: ¿qué porcentaje de los pedidos se cancela? ¿Cuál es el método de pago más usado?

In [11]:
# Estos conteos parten de TODOS los pedidos limpios, no solo los entregados.
conteo_estados = df_limpio.groupBy("estado").count().orderBy(F.desc("count"), "estado")
conteo_pagos = df_limpio.groupBy("metodo_pago").count().orderBy(F.desc("count"), "metodo_pago")
conteo_estados.show()
conteo_pagos.show()

n_cancelados = df_limpio.filter(F.col("estado") == "CANCELADO").count()
porcentaje_cancelados = 100 * n_cancelados / n_limpios
print(f"Cancelaciones: {n_cancelados} / {n_limpios} x 100 = {porcentaje_cancelados:.2f}%")

# Respuesta: se cancela el 9.78% de los pedidos limpios (5 339 / 54 566).
# Tarjeta es el método más usado, con 21 958 pedidos.
# Los 5 616 cancelados de la Parte 1 incluían duplicados y filas inválidas.

+---------+-----+
|   estado|count|
+---------+-----+
|ENTREGADO|45665|
|CANCELADO| 5339|
| DEVUELTO| 3562|
+---------+-----+

+-------------+-----+
|  metodo_pago|count|
+-------------+-----+
|      Tarjeta|21958|
|         Yape|16287|
|     Efectivo| 8211|
|Transferencia| 8110|
+-------------+-----+

Cancelaciones: 5339 / 54566 x 100 = 9.78%


### ✏️ Ejercicio 2.6 — Ventas por ciudad (2 pts)
Con los pedidos **entregados**, crea el DataFrame `ventas_por_ciudad` con una fila por ciudad y estas columnas:
- `pedidos`: número de pedidos
- `unidades`: suma de `cantidad`
- `ventas`: suma de `total` (en decimal, sin notación científica)
- `ticket_promedio`: promedio de `total` redondeado a 2 decimales

Ordénalo de mayor a menor `ventas` y muéstralo. Responde: ¿qué ciudad vende más y cuál menos?

In [12]:
# Una fila por ciudad y cuatro indicadores, solo sobre pedidos entregados.
ventas_por_ciudad = (entregados.groupBy("ciudad")
    .agg(
        F.count("id_pedido").alias("pedidos"),
        F.sum("cantidad").alias("unidades"),
        F.sum("total").cast("decimal(18,2)").alias("ventas"),
        F.round(F.avg("total"), 2).cast("decimal(18,2)").alias("ticket_promedio")
    )
    .orderBy(F.desc("ventas"), "ciudad"))
ventas_por_ciudad.show(truncate=False)

# Respuesta: Lima vende más (S/ 31 123 284.87) y Chiclayo menos
# (S/ 7 766 903.83). El ticket promedio es ventas / pedidos, no ventas / unidades.
# El resultado tiene 5 columnas en total: ciudad + los 4 indicadores pedidos.

+--------+-------+--------+-----------+---------------+
|ciudad  |pedidos|unidades|ventas     |ticket_promedio|
+--------+-------+--------+-----------+---------------+
|Lima    |18341  |45793   |31123284.87|1696.92        |
|Arequipa|6760   |16824   |11215911.73|1659.16        |
|Cusco   |5961   |14913   |9804943.29 |1644.85        |
|Trujillo|5483   |13648   |8951107.45 |1632.52        |
|Piura   |4554   |11480   |7852829.32 |1724.38        |
|Chiclayo|4566   |11282   |7766903.83 |1701.03        |
+--------+-------+--------+-----------+---------------+



### ✏️ Ejercicio 2.7 — Las mismas preguntas con Spark SQL (1.5 pts)
**Este ejercicio es obligatorio con SQL.**
1. Registra `df_limpio` como la vista temporal `pedidos_limpios`.
2. Con SQL, obtén por `categoria` el número de pedidos **entregados** y la suma de sus ventas, de mayor a menor.
3. Con SQL, obtén por `categoria` el número de pedidos **devueltos**, de mayor a menor. Responde: ¿qué categoría tiene más devoluciones?

In [13]:
# La vista permite consultar el DataFrame limpio con SQL.
df_limpio.createOrReplaceTempView("pedidos_limpios")

# Consulta 1: ventas y número de pedidos entregados por categoría.
ventas_categoria = spark.sql("""
    SELECT categoria,
           COUNT(*) AS pedidos_entregados,
           CAST(SUM(total) AS DECIMAL(18,2)) AS ventas
    FROM pedidos_limpios
    WHERE estado = 'ENTREGADO'
    GROUP BY categoria
    ORDER BY ventas DESC, categoria
""")
ventas_categoria.show(truncate=False)

# Consulta 2: número de pedidos devueltos por categoría.
devoluciones_categoria = spark.sql("""
    SELECT categoria, COUNT(*) AS pedidos_devueltos
    FROM pedidos_limpios
    WHERE estado = 'DEVUELTO'
    GROUP BY categoria
    ORDER BY pedidos_devueltos DESC, categoria
""")
devoluciones_categoria.show(truncate=False)

# Respuesta: Tecnologia lidera las ventas entregadas: S/ 59 937 691.17.
# Moda tiene más devoluciones: 2 340, frente a 873 de Tecnologia y 349 de Hogar.
# Son cantidades de pedidos devueltos; no porcentajes de devolución.

+----------+------------------+-----------+
|categoria |pedidos_entregados|ventas     |
+----------+------------------+-----------+
|Tecnologia|20216             |59937691.17|
|Moda      |15713             |10007489.55|
|Hogar     |9736              |6769799.77 |
+----------+------------------+-----------+

+----------+-----------------+
|categoria |pedidos_devueltos|
+----------+-----------------+
|Moda      |2340             |
|Tecnologia|873              |
|Hogar     |349              |
+----------+-----------------+



### ✏️ Ejercicio 2.8 — Guardar el resultado (1 pts)
1. Guarda `ventas_por_ciudad` como CSV con cabecera en `resultados/ventas_por_ciudad_spark`, en **un solo archivo** (pista: `coalesce(1)` y `mode("overwrite")`).
2. Lista la carpeta que creó Spark. Responde: ¿por qué Spark crea una **carpeta** y no un archivo?

In [14]:
# Se unifican las seis filas del resumen en una partición para escribir un CSV.
(ventas_por_ciudad.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv(f"{RESULTADOS}/ventas_por_ciudad_spark"))

!ls -lh resultados/ventas_por_ciudad_spark

# Respuesta: Spark escribe un directorio porque puede producir un archivo
# de datos por partición y archivos auxiliares como _SUCCESS.
# coalesce(1) deja un único part-*.csv; no elimina la carpeta contenedora.

total 4.0K
-rw-r--r-- 1 root root 274 Nov 13 00:22 part-00000-5f6f4ecd-2f29-4593-8ab6-2ad481d6bdcc-c000.csv
-rw-r--r-- 1 root root   0 Nov 13 00:22 _SUCCESS


## Parte 3 — Cerrar el proceso (Línea de comandos · 2 pts)
Spark ya terminó. Ahora deja las carpetas ordenadas para el próximo mes: lo que está en `raw/` ya se procesó y el resultado debe tener un nombre claro.

### ✏️ Ejercicio 3.1 — Mover los CSV procesados (1 pts)
1. Mueve todos los CSV de `datos/raw` a `datos/procesados` con `mv`.
2. Comprueba que `datos/raw` quedó vacía y cuenta cuántos archivos hay en `datos/procesados`.
3. Responde: ¿por qué es importante sacar los archivos de `raw/` después de procesarlos?

In [15]:
%%bash
# Mover después de guardar el resultado: Spark todavía podría leer raw antes.
mv datos/raw/*.csv datos/procesados/
ls -lh datos/raw
find datos/raw -type f | wc -l
find datos/procesados -type f -name '*.csv' | wc -l

# Respuesta: raw queda con 0 archivos; procesados contiene 29 CSV.
# Así diferenciamos los archivos pendientes de los ya tratados y evitamos
# volver a incluir el mismo lote por error. Conservarlos permite auditoría.

total 0
0
29


### ✏️ Ejercicio 3.2 — Dejar el resultado con un nombre claro (1 pts)
1. Mueve el archivo `part-*.csv` que creó Spark a `resultados/ventas_por_ciudad.csv` (con `mv` también se renombra).
2. Borra la carpeta `resultados/ventas_por_ciudad_spark`.
3. Muestra el contenido de `resultados/ventas_por_ciudad.csv` y la estructura final con `find datos resultados -type f | sort`.

In [16]:
%%bash
# mv mueve el archivo y le asigna el nombre solicitado.
mv resultados/ventas_por_ciudad_spark/part-*.csv resultados/ventas_por_ciudad.csv

# El CSV ya está fuera; se retira solo su carpeta temporal de Spark.
rm -r resultados/ventas_por_ciudad_spark

cat resultados/ventas_por_ciudad.csv
find datos resultados -type f | sort

# Resultado final: 29 CSV procesados, 2 vacíos en cuarentena,
# raw vacía y resultados/ventas_por_ciudad.csv con 6 filas más cabecera.

ciudad,pedidos,unidades,ventas,ticket_promedio
Lima,18341,45793,31123284.87,1696.92
Arequipa,6760,16824,11215911.73,1659.16
Cusco,5961,14913,9804943.29,1644.85
Trujillo,5483,13648,8951107.45,1632.52
Piura,4554,11480,7852829.32,1724.38
Chiclayo,4566,11282,7766903.83,1701.03
datos/cuarentena/pedidos_2026-10-12.csv
datos/cuarentena/pedidos_2026-10-25.csv
datos/procesados/pedidos_2026-10-01.csv
datos/procesados/pedidos_2026-10-02.csv
datos/procesados/pedidos_2026-10-03.csv
datos/procesados/pedidos_2026-10-04.csv
datos/procesados/pedidos_2026-10-05.csv
datos/procesados/pedidos_2026-10-06.csv
datos/procesados/pedidos_2026-10-07.csv
datos/procesados/pedidos_2026-10-08.csv
datos/procesados/pedidos_2026-10-09.csv
datos/procesados/pedidos_2026-10-10.csv
datos/procesados/pedidos_2026-10-11.csv
datos/procesados/pedidos_2026-10-13.csv
datos/procesados/pedidos_2026-10-14.csv
datos/procesados/pedidos_2026-10-15.csv
datos/procesados/pedidos_2026-10-16.csv
datos/procesados/pedidos_2026-10-17.csv
datos/

## Parte 4 — Entregar con Git (2.5 pts)
Esta parte se hace **en la terminal de tu computadora** (Git Bash en Windows, Terminal en Mac/Linux), siguiendo los pasos del documento del trabajo final.

1. Descarga el resultado ejecutando la celda de abajo (se guarda en tu carpeta de Descargas).
2. Escribe tu nombre y el link de tu repositorio al inicio de este notebook.
3. **Entorno de ejecución → Reiniciar y ejecutar todo**, y luego **Archivo → Descargar → Descargar .ipynb**.
4. Sube los dos archivos a tu repositorio público con `git add`, `git commit` y `git push`.

**Pendiente de realizar en tu computadora:** crear el repositorio público `trabajo-final-spark`, completar su URL arriba y hacer tu propio commit y push. Consulta los comandos de la guía. Esta versión todavía no está publicada.

**Conclusión:** sobre 54 566 pedidos limpios, 45 665 fueron entregados, 5 339 cancelados y 3 562 devueltos. Las ventas entregadas suman S/ 76 714 980.49. Lima lidera las ventas y Moda el número de devoluciones. Los archivos del 12 y 25 están vacíos: el análisis corresponde a los datos disponibles de 29 días, y no acredita ventas completas de todo el mes.

In [17]:
# En Colab esta celda descarga el CSV. Fuera de Colab muestra su ubicación.
try:
    from google.colab import files
except ImportError:
    print("CSV generado:", f"{RESULTADOS}/ventas_por_ciudad.csv")
    print("Para descargarlo desde Colab, ejecuta esta misma celda allí.")
else:
    files.download(f"{RESULTADOS}/ventas_por_ciudad.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>